# POLYDIM: ensayo diagnóstico con el mismo trabajo, NO certificación PMTP multiagente
Comparación de `generate()` con input IDs frente a `generate()` desde embeddings reconstruidos por Stiefel. Resultados reales solo al ejecutarlo en Colab.


In [ ]:
%pip install -q torch transformers safetensors huggingface_hub accelerate


In [ ]:
"""CONTROL DE INFERENCIA HF: texto vs inputs_embeds con misma tarea.
NO es un test de comunicación entre dos modelos ni de PMTP.
Colab GPU PyTorch; ejecutar con pesos públicos descargables.
Dependencias: pip install -q torch transformers safetensors huggingface_hub accelerate
"""
import hashlib
import json
import platform
import random
import statistics
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

MODEL_ID = 'Qwen/Qwen2.5-0.5B'
ADAPTER_REPO = 'polydim-cla/polydim-stiefel-manifold-adapters'
ADAPTER_FILE = '2_qwen2_5_PMTP_v1/adapter_weights.safetensors'
PROMPT = 'Escribe una función Python que calcule la suma de dos enteros.'
MAX_NEW = 24
WARMUP = 1
REPEATS = 5
SEED = 20261018

torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

def synchronize():
    if device == 'cuda': torch.cuda.synchronize()

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
path = hf_hub_download(repo_id=ADAPTER_REPO,filename=ADAPTER_FILE)
weights = load_file(path)
if 'W' not in weights:
    raise RuntimeError(f"No existe tensor W. Encontrados: {list(weights)}")
W_cpu = weights['W'].detach().cpu().to(dtype=torch.float64)
hidden = int(model.config.hidden_size)
if W_cpu.ndim != 2 or W_cpu.shape[1] != hidden or W_cpu.shape[0] < hidden:
    raise RuntimeError(f"La ruta isometrica necesita W=(n,{hidden}), n>={hidden}; se encontro {tuple(W_cpu.shape)}")
err = (W_cpu.T @ W_cpu - torch.eye(hidden,dtype=torch.float64)).abs().max().item()
print('Defecto columnas de W, max abs:',err)
if err > 1e-3:
    raise RuntimeError('No es un adaptador de columnas aproximadamente ortonormales: detener ensayo')
U = W_cpu.to(device=device,dtype=dtype)

def inference(arm):
    # AMBAS rutas cuentan tokenizer, proyección (si la hay), generación y decodificación.
    batch = tokenizer(PROMPT, return_tensors='pt').to(device)
    if arm == 'A':
        out = model.generate(input_ids=batch['input_ids'],attention_mask=batch['attention_mask'],
                             do_sample=False,max_new_tokens=MAX_NEW,pad_token_id=tokenizer.eos_token_id)
        generated = out[0, batch['input_ids'].shape[1]:]
    elif arm == 'C_roundtrip':
        # Embedding autentico del MISMO texto, elevación y recuperación mediante U.
        # NO simula transmisión entre cerebros/modelos; NO usa PMTP.
        embeddings = model.get_input_embeddings()(batch['input_ids'])
        z_high = embeddings @ U.T
        recovered = z_high @ U
        out = model.generate(inputs_embeds=recovered,attention_mask=batch['attention_mask'],
                             do_sample=False,max_new_tokens=MAX_NEW,pad_token_id=tokenizer.eos_token_id)
        # HF puede devolver IDs que contengan tokens ficticios de prompt.
        # Por tanto se registra como tokens generados candidatos y se audita manualmente.
        generated = out[0][-MAX_NEW:]
    else: raise ValueError(arm)
    text = tokenizer.decode(generated,skip_special_tokens=True)
    return {'text': text, 'output_id_count':int(out.shape[-1]),'decoded_count':int(generated.shape[-1])}

runs=[]
for rep in range(-WARMUP,REPEATS):
    arms=['A','C_roundtrip']
    random.shuffle(arms)
    for arm in arms:
        synchronize();start=time.perf_counter_ns()
        with torch.inference_mode(): out=inference(arm)
        synchronize();elapsed=(time.perf_counter_ns()-start)/1_000_000
        if rep >= 0:
            runs.append({'trial':rep,'arm':arm,'latency_ms':elapsed, **out})
            print(arm,'trial',rep,'time_ms',round(elapsed,3),'response:',out['text'][:120].replace('\n',' '))

latencies={arm:[r['latency_ms'] for r in runs if r['arm']==arm] for arm in ['A','C_roundtrip']}
report={
  'experiment':'SAME_MODEL_SAME_TEXT_SAME_MAX_NEW_TEXT_IDS_VS_STIEFEL_ROUNDTRIP_EMBEDS',
  'not_covered':['MULTI_MODEL','INTERPROCESS_PMTP','LATENT_MIND_TO_MIND','BENCHMARK_ACCURACY'],
  'device':device, 'device_name':torch.cuda.get_device_name(0) if device=='cuda' else platform.processor(),
  'model':MODEL_ID,'adapter_repo':ADAPTER_REPO,'adapter_file':ADAPTER_FILE,
  'adapter_sha256':hashlib.sha256(open(path,'rb').read()).hexdigest(),
  'ortho_defect':err,'warmup':WARMUP,'repeats':REPEATS,
  'means_ms':{k:statistics.mean(v) for k,v in latencies.items()},
  'all_trials':runs,
  'warning':'No afirmar speedup multiagente/PMTP; exigir oraculos de exactitud real para publicar.'
}
report['A_over_C_latency_ratio']=(report['means_ms']['A']/report['means_ms']['C_roundtrip'])
with open('polydim_colab_control.json','w',encoding='utf-8') as f:json.dump(report,f,ensure_ascii=False,indent=2)
print('Archivo: polydim_colab_control.json; cociente A/C =',round(report['A_over_C_latency_ratio'],3))
